# 02 — Qualité des données Citiz : qu'est-ce qui est exploitable ?

**Rôle : analyse.** Ne produit aucune table du pipeline : il justifie les colonnes retenues par le notebook `lakehouse` 01 (résumé en section 6).

**Objectif** : avant de construire quoi que ce soit, mesurer ce que contiennent vraiment les flux GBFS Citiz, colonne par colonne, et décider ce qu'on peut utiliser.

Après un aperçu des fichiers bruts, on répond à cinq questions :
1. Quelles colonnes sont remplies ? (taux de remplissage)
2. Où sont les stations ? (couverture géographique, doublons entre flux)
3. Les disponibilités peuvent-elles servir à mesurer l'usage ? (fraîcheur de `last_reported`, profondeur d'historique)
4. Que dit la flotte ? (motorisations, autonomie)
5. Quel verdict pour chaque donnée ?

Entrée : les CSV de `citiz/data/raw/gbfs/` produits par le notebook 01. Sortie : `citiz/data/processed/data_quality_report.csv`.

In [1]:
from pathlib import Path

current_directory = Path.cwd().resolve()
project_directory = None
for directory in [current_directory, *current_directory.parents]:
    if (directory / "citiz" / "AGENTS.md").exists():
        project_directory = directory
        break
if project_directory is None:
    raise FileNotFoundError("Open this notebook from the repository directory.")

module_directory = project_directory / "citiz"
gbfs_directory = module_directory / "data" / "raw" / "gbfs"
boundaries_directory = module_directory / "data" / "raw" / "boundaries"
processed_directory = module_directory / "data" / "processed"
for directory in [gbfs_directory, boundaries_directory, processed_directory]:
    directory.mkdir(parents=True, exist_ok=True)

import json
import pandas as pd

ID_COLUMNS = {"station_id": "string", "vehicle_type_id": "string", "post_code": "string"}


def read_gbfs_csv(file_name):
    """Read a raw CSV, keeping identifiers as text ("01" must not become 1)."""
    return pd.read_csv(gbfs_directory / file_name, dtype=ID_COLUMNS)


stations = read_gbfs_csv("citiz_stations.csv")
status = read_gbfs_csv("citiz_station_status.csv")
vehicle_types = read_gbfs_csv("citiz_vehicle_types.csv")
feeds = read_gbfs_csv("citiz_flux.csv")
print(f"stations: {len(stations)} | status rows: {len(status)} | vehicle types: {len(vehicle_types)} | feeds: {len(feeds)}")

stations: 1615 | status rows: 6459 | vehicle types: 337 | feeds: 15


## 0. Aperçu des fichiers bruts

Les 6 premières lignes de chacun des 4 CSV produits par le notebook 01, pour voir leur forme avant de les mesurer.

In [2]:
raw_tables = {"citiz_flux": feeds, "citiz_stations": stations,
              "citiz_station_status": status, "citiz_vehicle_types": vehicle_types}
for name, table in raw_tables.items():
    print(f"{name}: {table.shape[0]} rows x {table.shape[1]} columns")
    display(table.head(6))

citiz_flux: 15 rows x 12 columns


,dataset_id,dataset_slug,dataset_title,publisher,page_url,licence,resource_title,gbfs_url,is_available,source,gbfs_version,feeds
0,678f5e10a7133882a3499576,citiz-autopartage-developpement,Autopartage Citiz Développement,Réseau Citiz,https://transport.data.gouv.fr/datasets/citiz-...,lov2,gbfs-citiz-dev,https://backend.citiz.fr/public/provider/12/gb...,True,provider_12,3.0,"gbfs,gbfs_versions,geofencing_zones,station_in..."
1,678e69be93b3f9e17df18b70,citiz-autopartage-auvergne-rhone-alpes,Autopartage Citiz Auvergne Rhône Alpes (AURA),Réseau Citiz,https://transport.data.gouv.fr/datasets/citiz-...,lov2,gbfs-citiz-autopartage-aura,https://backend.citiz.fr/public/provider/5/gbf...,True,provider_5,3.0,"gbfs,gbfs_versions,geofencing_zones,station_in..."
2,678e607cea68baffac0f0513,citiz-autopartage-grand-est,Autopartage Citiz Grand Est,Réseau Citiz,https://transport.data.gouv.fr/datasets/citiz-...,lov2,gbfs-citiz-autopartage-grand-est,https://backend.citiz.fr/public/provider/1/gbf...,True,provider_1,3.0,"gbfs,gbfs_versions,geofencing_zones,station_in..."
3,68017aade8f8b0f5ced063dc,offre-et-disponibilite-temps-reel-du-service-c...,Autopartage Citiz Métropole de Lyon,Métropole de Lyon,https://transport.data.gouv.fr/datasets/offre-...,lov2,Offre et disponibilité temps réel du service C...,https://data.grandlyon.com/files/rdata/lpa_mob...,True,offre-et-disponibilite-temps-reel-du-service-c...,3.0,"gbfs,gbfs_versions,geofencing_zones,station_in..."
4,678e6c068a2785ad5b2099f8,citiz-autopartage-provence,Autopartage Citiz Provence,Réseau Citiz,https://transport.data.gouv.fr/datasets/citiz-...,lov2,gbfs-citiz-autopartage-provence,https://backend.citiz.fr/public/provider/6/gbf...,True,provider_6,3.0,"gbfs,gbfs_versions,geofencing_zones,station_in..."
5,678e6dfe20c290b9271d7a1d,citiz-autopartage-occitanie,Autopartage Citiz Occitanie,Réseau Citiz,https://transport.data.gouv.fr/datasets/citiz-...,lov2,gbfs-citiz-autopartage-occitanie,https://backend.citiz.fr/public/provider/8/gbf...,True,provider_8,3.0,"gbfs,gbfs_versions,geofencing_zones,station_in..."


citiz_stations: 1615 rows x 18 columns


,source,station_id,name,short_name,lat,lon,address,post_code,city,capacity,is_virtual_station,vehicle_types_capacity,sources,station_uid,extracted_at,num_vehicles_available,is_renting,geometry_wkt
0,provider_12,43,Halles,NaN,47.392776,0.679685,34 Rue de la Grosse Tour,<NA>,NaN,3,False,NaN,provider_12,provider_12:43,2026-10-20 11:27:10.315374+00:00,2,True,POINT(0.6796849966049194 47.39277648925781)
1,provider_12,44,Constantine-Commerce,NaN,47.395157,0.684126,NaN,<NA>,NaN,2,False,NaN,provider_12,provider_12:44,2026-10-20 11:27:10.315374+00:00,2,True,POINT(0.6841260194778442 47.39515686035156)
2,provider_12,45,Cathédrale-Sicard,NaN,47.394527,0.693761,17 place François Sicard,<NA>,NaN,2,False,NaN,provider_12,provider_12:45,2026-10-20 11:27:10.315374+00:00,2,True,POINT(0.6937609910964966 47.394527435302734)
3,provider_12,46,Gare,NaN,47.390404,0.694493,NaN,<NA>,NaN,2,False,NaN,provider_12,provider_12:46,2026-10-20 11:27:10.315374+00:00,2,True,POINT(0.6944929957389832 47.390403747558594)
4,provider_12,47,Jean Jaurès (16 bd Beranger),NaN,47.389912,0.687188,NaN,<NA>,NaN,2,False,NaN,provider_12,provider_12:47,2026-10-20 11:27:10.315374+00:00,2,True,POINT(0.6871880292892456 47.38991165161133)
5,provider_12,48,Michelet (place),NaN,47.386826,0.691275,NaN,<NA>,NaN,3,False,NaN,provider_12,provider_12:48,2026-10-20 11:27:10.315374+00:00,2,True,POINT(0.6912750005722046 47.38682556152344)


citiz_station_status: 6459 rows x 12 columns


,source,station_id,num_vehicles_available,num_vehicles_disabled,num_docks_available,is_installed,is_renting,is_returning,last_reported,vehicle_types_available,snapshot_ts,station_uid
0,provider_12,43,2,NaN,0,True,True,True,2026-09-16 12:37:17.516500700+00:00,"[{""vehicle_type_id"": ""16"", ""count"": 1}, {""vehi...",2026-10-19 14:43:54.580579+00:00,provider_12:43
1,provider_12,44,2,NaN,0,True,True,True,2025-03-21 11:42:13.271082600+00:00,"[{""vehicle_type_id"": ""16"", ""count"": 1}, {""vehi...",2026-10-19 14:43:54.580579+00:00,provider_12:44
2,provider_12,45,2,NaN,0,True,True,True,2026-08-17 07:57:16.910994+00:00,"[{""vehicle_type_id"": ""1"", ""count"": 1}, {""vehic...",2026-10-19 14:43:54.580579+00:00,provider_12:45
3,provider_12,46,2,NaN,0,True,True,True,2025-06-04 14:20:35.609229400+00:00,"[{""vehicle_type_id"": ""1"", ""count"": 1}, {""vehic...",2026-10-19 14:43:54.580579+00:00,provider_12:46
4,provider_12,47,2,NaN,0,True,True,True,2025-03-21 11:41:47.179023700+00:00,"[{""vehicle_type_id"": ""16"", ""count"": 1}, {""vehi...",2026-10-19 14:43:54.580579+00:00,provider_12:47
5,provider_12,48,2,NaN,0,True,True,True,2025-04-03 03:45:35.183462700+00:00,"[{""vehicle_type_id"": ""4"", ""count"": 1}, {""vehic...",2026-10-19 14:43:54.580579+00:00,provider_12:48


citiz_vehicle_types: 337 rows x 9 columns


,source,vehicle_type_id,name,make,model,form_factor,propulsion_type,rider_capacity,max_range_meters
0,provider_12,1,Renault Twingo,Renault,Twingo,car,combustion,NaN,0
1,provider_12,2,Renault Clio V,Renault,Clio V,car,combustion,NaN,0
2,provider_12,3,Renault Zoé,Renault,Zoé,car,electric,NaN,0
3,provider_12,4,Renault Kangoo,Renault,Kangoo,car,combustion,NaN,0
4,provider_12,11,Toyota Yaris,Toyota,Yaris,car,hybrid,NaN,0
5,provider_12,13,Suzuki Swift,Suzuki,Swift,car,hybrid,NaN,0


## 1. Taux de remplissage par colonne

Pour chaque colonne : part des lignes renseignées et nombre de valeurs distinctes. Une colonne vide partout ou constante n'apporte aucune information.

Règle de verdict automatique :
- **vide** : aucune valeur ;
- **constante** : une seule valeur (pas d'information) ;
- **partielle** : moins de 80 % de lignes renseignées ;
- **exploitable** : sinon.

Ce verdict est un premier tri, complété par les analyses suivantes.

In [3]:
def profile(df, table):
    rows = []
    for column in df.columns:
        filled = df[column].notna().mean()
        distinct = df[column].nunique()
        if filled == 0:
            verdict = "vide"
        elif distinct == 1:
            verdict = "constante"
        elif filled < 0.8:
            verdict = "partielle"
        else:
            verdict = "exploitable"
        rows.append({"table": table, "column": column, "fill_rate": round(filled, 3),
                     "distinct_values": distinct, "auto_verdict": verdict})
    return pd.DataFrame(rows)


quality = pd.concat([
    profile(stations, "citiz_stations"),
    profile(status, "citiz_station_status"),
    profile(vehicle_types, "citiz_vehicle_types"),
], ignore_index=True)
quality[quality.auto_verdict != "exploitable"]

,table,column,fill_rate,distinct_values,auto_verdict
3,citiz_stations,short_name,0.00,0,vide
6,citiz_stations,address,0.67,1080,partielle
7,citiz_stations,post_code,0.00,0,vide
8,citiz_stations,city,0.00,0,vide
11,citiz_stations,vehicle_types_capacity,0.00,0,vide
14,citiz_stations,extracted_at,1.00,1,constante
16,citiz_stations,is_renting,1.00,1,constante
21,citiz_station_status,num_vehicles_disabled,0.00,0,vide
22,citiz_station_status,num_docks_available,1.00,1,constante
24,citiz_station_status,is_renting,1.00,1,constante


## 2. Couverture géographique

- Répartition des stations par réseau.
- Stations présentes dans plusieurs flux (`sources` contient une virgule) : doublons détectés à l'extraction.
- Stations en Outre-mer : le sujet demande d'inclure les DROM.

In [4]:
network_names = feeds.set_index("source").dataset_title.str.replace("Autopartage Citiz ", "", regex=False)
by_network = stations.groupby("source").size().rename("stations").to_frame()
by_network["network"] = network_names
by_network = by_network.sort_values("stations", ascending=False)

multi_feed = stations.sources.str.contains(",", na=False).sum()
overseas = (stations.lat < 41).sum()
print(f"Stations listed in several feeds: {multi_feed}")
print(f"Overseas stations (DROM): {overseas}")
print(f"Virtual stations (no physical spot): {stations.is_virtual_station.sum()}")
by_network

Stations listed in several feeds: 0
Overseas stations (DROM): 0
Virtual stations (no physical spot): 57


,stations,network
source,,
provider_5,412,Auvergne Rhône Alpes (AURA)
provider_1,290,Grand Est
offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon,200,Métropole de Lyon
provider_8,142,Occitanie
provider_4,100,Gironde
provider_6,97,Provence
provider_16,80,Haut de France
provider_19,66,Rennes Métropole
provider_12,64,Développement


In [5]:
print("Capacity (cars per station):")
print(stations.capacity.describe().round(1).to_string())
stations.nlargest(5, "capacity")[["station_uid", "name", "capacity", "is_virtual_station"]]

Capacity (cars per station):
count    1615.0
mean        2.5
std         3.8
min         0.0
25%         2.0
50%         2.0
75%         3.0
max       103.0


,station_uid,name,capacity,is_virtual_station
326,provider_5:1027,Véhicules hors-circuit (Les véhicules sur cet...,103,False
139,provider_5:795,DEMOBOX (Réservé Citiz),100,False
1028,provider_6:1442,Endoume,25,False
766,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 1er - Parking LPA Terreaux,12,False
1213,provider_16:298,GARE LILLE FLANDRES,12,False


### Pseudo-stations internes

Certaines lignes ne sont pas des stations ouvertes au public : stockage de véhicules hors service, démonstration, livraison. Leur nom les signale (`ZZ_`, « hors-circuit », « DEMOBOX », « NE PAS RESERVER »). Elles faussent les capacités (jusqu'à 103 places) et doivent être **exclues des analyses**. Elles restent dans les fichiers bruts.

Les stations de capacité 0 sont aussi listées : elles existent dans le flux mais n'ont aucune place déclarée.

In [6]:
INTERNAL_NAME_PATTERN = r"^ZZ_|hors.circuit|DEMOBOX|NE PAS RESERVER"
internal = stations[stations.name.str.contains(INTERNAL_NAME_PATTERN, case=False, regex=True, na=False)]
print(f"Internal pseudo-stations: {len(internal)}")
print(f"Stations with capacity 0: {(stations.capacity == 0).sum()}")
internal[["station_uid", "name", "capacity"]]

Internal pseudo-stations: 3
Stations with capacity 0: 23


,station_uid,name,capacity
6,provider_12:49,ZZ_SXB LIVRAISON NE PAS RESERVER,5
139,provider_5:795,DEMOBOX (Réservé Citiz),100
326,provider_5:1027,Véhicules hors-circuit (Les véhicules sur cet...,103


## 3. Les disponibilités peuvent-elles mesurer l'usage ?

L'idée de départ : relever régulièrement le nombre de voitures disponibles et en déduire quand une station est vide, donc utilisée.

Deux conditions sont nécessaires :
- **un historique** : plusieurs instantanés dans le temps ;
- **des données fraîches** : `last_reported` indique la dernière mise à jour du statut par l'opérateur. S'il date de plusieurs mois, le flux ne reflète pas les mouvements réels des voitures.

`age_days` = écart entre l'instantané et `last_reported`.

In [7]:
snapshot = pd.to_datetime(status.snapshot_ts, utc=True, format="ISO8601")
last_reported = pd.to_datetime(status.last_reported, utc=True, format="ISO8601")
status["age_days"] = (snapshot - last_reported).dt.days

print(f"Snapshots collected: {status.snapshot_ts.nunique()}")
print(f"Status older than 30 days: {(status.age_days > 30).mean():.0%}")
print(f"Status older than 1 year: {(status.age_days > 365).mean():.0%}")
print(f"Stations with no car available: {(status.num_vehicles_available == 0).mean():.0%}")
status.age_days.describe().round(0)

Snapshots collected: 4
Status older than 30 days: 95%
Status older than 1 year: 57%
Stations with no car available: 7%


count    6459.0
mean      452.0
std       310.0
min         0.0
25%       167.0
50%       399.0
75%       732.0
max      1305.0
Name: age_days, dtype: float64

In [8]:
age_by_network = status.groupby("source").age_days.median().rename("median_age_days").to_frame()
age_by_network["network"] = network_names
age_by_network.sort_values("median_age_days")

,median_age_days,network
source,,
provider_7,115.5,Angers Loire Métropole
provider_8,185.5,Occitanie
provider_15,236.0,Nantes Métropole
provider_20,249.5,Pays Basque AUPA
offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon,265.5,Métropole de Lyon
provider_19,282.0,Rennes Métropole
provider_12,291.5,Développement
provider_9,306.0,Grand Poitiers
provider_1,396.0,Grand Est


## 4. La flotte

`vehicle_types` décrit les modèles par réseau. On regarde la motorisation (utile pour l'impact carbone) et l'autonomie.

In [9]:
print(vehicle_types.propulsion_type.value_counts().to_string())
print()
print(vehicle_types.form_factor.value_counts().to_string())
print()
print(f"max_range_meters equal to 0: {(vehicle_types.max_range_meters == 0).mean():.0%}")

propulsion_type
combustion    182
hybrid         87
electric       58
human          10

form_factor
car              329
other              6
cargo_bicycle      2

max_range_meters equal to 0: 96%


In [10]:
# Vehicles currently available, by propulsion (from the JSON column of station_status)
vehicle_rows = []
for row in status.itertuples():
    if isinstance(row.vehicle_types_available, str):
        for item in json.loads(row.vehicle_types_available):
            vehicle_rows.append({"source": row.source, "vehicle_type_id": str(item["vehicle_type_id"]),
                                 "count": item.get("count", 0)})
available = pd.DataFrame(vehicle_rows).merge(
    vehicle_types[["source", "vehicle_type_id", "propulsion_type"]], on=["source", "vehicle_type_id"], how="left")
available.groupby("propulsion_type", dropna=False)["count"].sum().sort_values(ascending=False)

propulsion_type
hybrid        4774
combustion    4747
electric       720
human           92
Name: count, dtype: int64

## 5. Verdict

On complète le verdict automatique par les constats ci-dessus. La colonne `note` explique la décision ; elle est reprise dans le README du module.

In [11]:
NOTES = {
    ("citiz_stations", "station_uid"): ("exploitable", "Clé de jointure de l'équipe, unique."),
    ("citiz_stations", "lat"): ("exploitable", "Position WGS84, toujours renseignée."),
    ("citiz_stations", "lon"): ("exploitable", "Position WGS84, toujours renseignée."),
    ("citiz_stations", "capacity"): ("exploitable", "Nombre de places ; exclure les 3 pseudo-stations internes (jusqu'à 103 places)."),
    ("citiz_stations", "name"): ("exploitable", "Permet aussi de repérer les pseudo-stations internes (ZZ_, hors-circuit, DEMOBOX)."),
    ("citiz_station_status", "num_docks_available"): ("non exploitable", "Toujours 0 : notion de borne propre au vélo, sans objet en autopartage."),
    ("citiz_station_status", "is_renting"): ("non exploitable", "Toujours vrai : aucune information."),
    ("citiz_station_status", "vehicle_types_available"): ("exploitable", "Détail des voitures disponibles par modèle (JSON)."),
    ("citiz_stations", "address"): ("partielle", "Absente pour environ un tiers des stations ; à compléter par géocodage inverse si besoin."),
    ("citiz_stations", "is_virtual_station"): ("exploitable", "Distingue les zones sans place physique."),
    ("citiz_stations", "num_vehicles_available"): ("à utiliser avec prudence", "Instantané unique, voir citiz_station_status."),
    ("citiz_station_status", "last_reported"): ("non exploitable pour l'usage", "Majoritairement vieux de plusieurs mois : ne reflète pas les mouvements réels."),
    ("citiz_station_status", "num_vehicles_available"): ("à valider", "Utilisable seulement avec un historique dense et des statuts frais."),
    ("citiz_vehicle_types", "propulsion_type"): ("exploitable", "Motorisation : base du calcul d'impact carbone."),
    ("citiz_vehicle_types", "max_range_meters"): ("non exploitable", "Vaut 0 pour presque tous les modèles."),
}
quality["final_verdict"] = quality.auto_verdict
quality["note"] = ""
for (table, column), (verdict, note) in NOTES.items():
    mask = (quality.table == table) & (quality.column == column)
    quality.loc[mask, "final_verdict"] = verdict
    quality.loc[mask, "note"] = note

quality.to_csv(processed_directory / "data_quality_report.csv", index=False)
print(f"Report -> {(processed_directory / 'data_quality_report.csv').relative_to(project_directory)}")
quality[["table", "column", "fill_rate", "final_verdict", "note"]]

Report -> citiz/data/processed/data_quality_report.csv


,table,column,fill_rate,final_verdict,note
0,citiz_stations,source,1.00,exploitable,
1,citiz_stations,station_id,1.00,exploitable,
2,citiz_stations,name,1.00,exploitable,Permet aussi de repérer les pseudo-stations in...
3,citiz_stations,short_name,0.00,vide,
4,citiz_stations,lat,1.00,exploitable,"Position WGS84, toujours renseignée."
5,citiz_stations,lon,1.00,exploitable,"Position WGS84, toujours renseignée."
6,citiz_stations,address,0.67,partielle,Absente pour environ un tiers des stations ; à...
7,citiz_stations,post_code,0.00,vide,
8,citiz_stations,city,0.00,vide,
9,citiz_stations,capacity,1.00,exploitable,Nombre de places ; exclure les 3 pseudo-statio...


## 6. Ce qui entre dans le pipeline

Le notebook `lakehouse/notebooks/01_build_lakehouse_tables.ipynb` applique ce verdict. Les tables **principales** servent au sujet ; les tables **annexes** sont conservées pour accumuler l'historique, sans être une mesure d'usage.

| Table Lakehouse | Rôle | Colonnes retenues | Colonnes écartées |
|---|---|---|---|
| `citiz_station` | principale | `station_uid`, position, `name`, `address` (partielle), `capacity`, `is_virtual_station`, `sources`, département, région, `is_internal` (pseudo-station) | `short_name`, `post_code`, `city`, `vehicle_types_capacity` (vides) ; `num_vehicles_available`, `is_renting` (état instantané) |
| `citiz_vehicle_type` | principale | marque, modèle, `form_factor`, `propulsion_type` | `rider_capacity` (vide), `max_range_meters` (0 pour 96 % des modèles) |
| `citiz_network`, `geo_region`, `geo_department`, `cost_parameter` | principale | toutes | — |
| `station_feature_aura` | principale | indicateurs OSM de `regional-data/` (27 colonnes) | doublons de `citiz_station`, colonnes vides, constantes ou non exploitables (23 colonnes) |
| `population_grid_aura` | principale | toutes les colonnes de `population/`, plus contour WKT et département | — |
| `citiz_station_status` | annexe | `num_vehicles_available`, `is_installed`, `is_returning`, `last_reported`, `status_age_days`, `is_stale` | `num_vehicles_disabled` (vide), `num_docks_available` (toujours 0), `is_renting` (toujours vrai) |
| `citiz_station_status_vehicle` | annexe | voitures disponibles par modèle | — |

Les pseudo-stations internes ne sont pas supprimées : elles sont signalées par `is_internal`, pour garder les mêmes `station_uid` que la référence partagée. Les filtrer avant toute analyse.